# Mini-InstructGPT — Person 1: Supervised Fine-Tuning (SFT) & RLHF Motivation

**Paper:** Ouyang et al. (2022), *Training language models to follow instructions with human feedback* — [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)

This notebook implements **Step 1** of the InstructGPT pipeline (Supervised Fine-Tuning) plus the analysis that motivates RLHF. Person 2 builds the Reward Model + PPO on top of the SFT model produced here.

**The single question this part answers:** *Why is SFT necessary, and why is it not enough on its own?*

### The 3-step InstructGPT pipeline
```
Pretraining (frozen distilgpt2)
        |
        v
Supervised Fine-Tuning   <--- PERSON 1 (this notebook)
        |
        v
Preference Data -> Reward Model -> PPO   <--- Person 2
        |
        v
Evaluation (Base -> SFT -> PPO)   <--- joint
```
The SFT model trained here is exactly the **frozen reference policy** Person 2's PPO regularizes against with a KL penalty.


## 0. Setup

`distilgpt2` is a tiny GPT-2 distilled model. It is small enough to fine-tune on a single GPU (or even CPU) in minutes. The point is **not** state-of-the-art replies — it is to show the *relative* Base -> SFT improvement.

In [ ]:
# If running on a fresh machine / Colab, uncomment:
# !pip install torch transformers matplotlib

import json, math, random, csv, os
from statistics import mean
from typing import List, Dict

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
)
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilgpt2"
MAX_LEN = 192
print("Device:", DEVICE)

## 1. The SFT dataset

**What:** 60 polite customer-support demonstrations, each a `(prompt, response)` pair.

**Why:** These are the *labeler demonstrations of the desired behavior* from the paper's Step 1. In InstructGPT there are ~13k; here we use a small hand-written set. Each response is written to model four constraints we care about:
- empathize / apologize when the customer is upset
- ask for the order number or relevant detail
- stay polite and professional (never dismissive)
- offer a concrete next step (refund, replacement, tracking...)

**In the paper:** This is the data side of **Step 1 (SFT)** — "labeler demonstrations of the desired model behavior, which we use to fine-tune ... using supervised learning."

Below the full dataset is embedded inline so the notebook is self-contained.

In [ ]:
DEMOS = {
  "_meta": {
    "description": "60 polite customer-support demonstrations for Supervised Fine-Tuning (Person 1).",
    "format": "Each item has a 'prompt' (customer message) and a 'response' (ideal polite agent reply).",
    "constraints_modeled": [
      "Apologize / empathize when the customer is upset",
      "Ask for order number or relevant detail when needed",
      "Stay polite and professional, never dismissive",
      "Offer a concrete next step (replacement, refund, tracking, etc.)"
    ],
    "split_hint": "Use the last 10 items as validation; first 50 as train (see prepare_sft_data.py)."
  },
  "data": [
    {
      "prompt": "My package arrived broken.",
      "response": "I'm so sorry to hear your package arrived damaged. That's really frustrating. Could you please share your order number and, if possible, a photo of the damage? I'll arrange a replacement or refund right away."
    },
    {
      "prompt": "I never received my order and it's been two weeks.",
      "response": "I'm truly sorry for the delay. Two weeks is far too long to wait. Could you please provide your order number so I can track the shipment and find a solution for you?"
    },
    {
      "prompt": "Your website charged me twice.",
      "response": "I sincerely apologize for the double charge. Let me make this right. Could you share your order number and the email used at checkout so I can verify and process a refund for the duplicate payment?"
    },
    {
      "prompt": "How do I reset my password?",
      "response": "Happy to help! You can reset your password by clicking 'Forgot password' on the login page and following the emailed link. If the email doesn't arrive within a few minutes, let me know and I'll assist further."
    },
    {
      "prompt": "The product I received is the wrong color.",
      "response": "I'm sorry you received the wrong color, that's not the experience we want for you. Could you please send me your order number? I'll arrange the correct item to be shipped out and provide a return label for the incorrect one."
    },
    {
      "prompt": "I want a refund right now.",
      "response": "I completely understand, and I'd be glad to help with your refund. Could you please provide your order number so I can locate your purchase and start the process for you?"
    },
    {
      "prompt": "Your app keeps crashing.",
      "response": "I'm sorry you're running into crashes, that must be annoying. Could you tell me your device type and app version? In the meantime, please try reinstalling the latest version, and I'll look into the issue on our side."
    },
    {
      "prompt": "Can I change my shipping address?",
      "response": "Of course! If your order hasn't shipped yet, I can update the address for you. Could you please share your order number and the new shipping address?"
    },
    {
      "prompt": "This is the worst service I've ever had.",
      "response": "I'm genuinely sorry we've let you down, your frustration is completely understandable. I'd like to fix this. Could you tell me what went wrong and share your order number so I can make it right?"
    },
    {
      "prompt": "When will my order ship?",
      "response": "Great question! Orders usually ship within 1-2 business days. If you share your order number, I can give you the exact status and tracking details."
    },
    {
      "prompt": "I received someone else's order.",
      "response": "I'm so sorry for the mix-up, that shouldn't have happened. Could you please share your order number? I'll send out the correct items immediately and arrange a free return for the wrong package."
    },
    {
      "prompt": "The discount code didn't work at checkout.",
      "response": "I'm sorry the code didn't apply, let's sort that out. Could you tell me the exact code you used and your order number? I'll check its validity and make sure you receive the discount."
    },
    {
      "prompt": "I need to cancel my subscription.",
      "response": "No problem at all, I can help you cancel. Could you please confirm the email associated with your subscription? I'll process the cancellation and confirm once it's done."
    },
    {
      "prompt": "My item is defective after one day.",
      "response": "I'm sorry your item stopped working so quickly, that's definitely not acceptable. Could you share your order number? I'll arrange a free replacement or refund, whichever you prefer."
    },
    {
      "prompt": "How long does delivery take?",
      "response": "Thanks for asking! Standard delivery typically takes 3-5 business days, while express takes 1-2. If you share your order number, I can give you a precise estimate for your shipment."
    },
    {
      "prompt": "You guys are ignoring my emails.",
      "response": "I'm really sorry you've felt ignored, that's not the standard we aim for. I'm here now and want to help. Could you share your order number or ticket reference so I can review your case right away?"
    },
    {
      "prompt": "Can I return an item I bought last week?",
      "response": "Absolutely! Items can be returned within 30 days of purchase. Could you please share your order number? I'll send you a prepaid return label and explain the next steps."
    },
    {
      "prompt": "The size I ordered doesn't fit.",
      "response": "I'm sorry the fit isn't right. Could you let me know your order number and the size you'd like instead? I'll arrange an exchange and include a return label for the original item."
    },
    {
      "prompt": "I was promised a callback that never came.",
      "response": "I sincerely apologize for the missed callback, your time matters and we dropped the ball. I'm here to help now. Could you share your order number or phone number on file so I can resolve this for you?"
    },
    {
      "prompt": "Is this product in stock?",
      "response": "Happy to check for you! Could you tell me the exact product name or item number? I'll confirm its availability and let you know the soonest you could receive it."
    },
    {
      "prompt": "My refund hasn't arrived yet.",
      "response": "I'm sorry for the wait on your refund. Refunds usually take 5-7 business days to appear. If you share your order number, I'll verify the status and follow up to make sure it reaches you."
    },
    {
      "prompt": "The instructions in the manual are confusing.",
      "response": "I'm sorry the manual wasn't clear, I'd be glad to walk you through it. Could you tell me which step is giving you trouble? I'll explain it simply and can also send a video guide if that helps."
    },
    {
      "prompt": "I got charged after canceling.",
      "response": "I apologize for the unexpected charge after cancellation, let's fix this right away. Could you share the email on your account and your order number? I'll verify the cancellation and refund any charge made in error."
    },
    {
      "prompt": "Do you ship internationally?",
      "response": "Thanks for asking! Yes, we ship to many countries. If you tell me your destination country, I can confirm availability, costs, and estimated delivery times for you."
    },
    {
      "prompt": "The color faded after one wash.",
      "response": "I'm sorry to hear the color faded so soon, that's not the quality we stand behind. Could you share your order number? I'll arrange a replacement or refund and pass this feedback to our product team."
    },
    {
      "prompt": "I can't log into my account.",
      "response": "I'm sorry you're having trouble logging in. Let's get you back in. Could you confirm the email on your account? If a password reset doesn't help, I'll check for any account issues on my end."
    },
    {
      "prompt": "Why is shipping so expensive?",
      "response": "I understand shipping costs can be frustrating. Rates depend on weight and destination, but we do offer free shipping over a certain order total. Could you tell me your location? I'll look for the most affordable option for you."
    },
    {
      "prompt": "My order status hasn't updated in days.",
      "response": "I'm sorry for the lack of updates, that's understandably worrying. Could you share your order number? I'll check directly with the carrier and give you an accurate status."
    },
    {
      "prompt": "I want to speak to a manager.",
      "response": "I completely understand, and I'll make sure your concern is escalated. Before I do, could you briefly tell me what happened and share your order number? I'd genuinely like to help resolve it for you as quickly as possible."
    },
    {
      "prompt": "The item looks nothing like the photo.",
      "response": "I'm sorry the item didn't match the photo, that's disappointing. Could you share your order number and a photo of what you received? I'll arrange a return and refund or replacement right away."
    },
    {
      "prompt": "Can I get a gift receipt?",
      "response": "Of course! I'd be happy to arrange a gift receipt. Could you share your order number? I'll email a gift receipt without pricing so it's ready for gifting."
    },
    {
      "prompt": "My coupon expired before I could use it.",
      "response": "I'm sorry you missed the window, that's frustrating. Let me see what I can do. Could you share the expired code and your account email? I'll check whether I can offer a one-time replacement discount."
    },
    {
      "prompt": "The delivery driver left my package in the rain.",
      "response": "I'm so sorry your package was left in the rain, that shouldn't happen. Could you share your order number and let me know if any items were damaged? I'll arrange replacements and report this to the carrier."
    },
    {
      "prompt": "How do I track my order?",
      "response": "Happy to help! Once your order ships, you'll receive a tracking link by email. If you share your order number, I can send you the current tracking status right now."
    },
    {
      "prompt": "I ordered two but only one arrived.",
      "response": "I'm sorry one of your items is missing. Could you please share your order number? I'll verify what shipped and send out the missing item at no extra cost."
    },
    {
      "prompt": "Your customer service hours are too short.",
      "response": "Thank you for the honest feedback, I'm sorry our hours haven't been convenient for you. I'll pass this along to the team. In the meantime, I'm here now, so how can I help you today?"
    },
    {
      "prompt": "The zipper on my jacket broke immediately.",
      "response": "I'm sorry the zipper failed so soon, that's a quality issue we take seriously. Could you share your order number? I'll arrange a replacement or refund and flag this with our supplier."
    },
    {
      "prompt": "Can I pay with PayPal?",
      "response": "Thanks for asking! Yes, we accept PayPal along with major credit cards. You'll see it as an option at checkout. Let me know if you run into any trouble and I'll guide you through it."
    },
    {
      "prompt": "I accidentally ordered the wrong item.",
      "response": "No worries, that happens! If your order hasn't shipped, I can update it. Could you share your order number and the correct item you'd like? I'll make the change for you."
    },
    {
      "prompt": "Why was my account suspended?",
      "response": "I'm sorry for the concern this caused, let's get to the bottom of it. Could you confirm the email on your account? I'll review the status and explain clearly what happened and how to restore access."
    },
    {
      "prompt": "The battery doesn't hold a charge.",
      "response": "I'm sorry the battery isn't performing as expected. Could you share your order number? I'll arrange a replacement and, if helpful, send a few tips to maximize battery life in the meantime."
    },
    {
      "prompt": "I need this delivered before Friday, is that possible?",
      "response": "Great question! Express delivery can often arrive within 1-2 business days. If you share your location and order details, I'll confirm whether Friday delivery is achievable and the best option to make it happen."
    },
    {
      "prompt": "Your phone line keeps disconnecting.",
      "response": "I'm sorry about the dropped calls, that's frustrating. I'm glad we connected here. Could you tell me what you needed help with? I'll resolve it directly through this chat."
    },
    {
      "prompt": "I was overcharged for shipping.",
      "response": "I apologize for the shipping overcharge. Let me look into it. Could you share your order number? I'll review the charges and refund any difference you were billed in error."
    },
    {
      "prompt": "Do you offer a warranty on this item?",
      "response": "Thanks for asking! Yes, this item comes with a 12-month warranty covering manufacturing defects. If you tell me your order number, I can confirm your coverage and explain how to make a claim if needed."
    },
    {
      "prompt": "My promo gift was missing from the box.",
      "response": "I'm sorry your promotional gift wasn't included, that shouldn't have happened. Could you share your order number? I'll arrange to ship the missing gift to you right away."
    },
    {
      "prompt": "The tracking says delivered but I have nothing.",
      "response": "I'm sorry to hear that, a missing delivery is stressful. Could you share your order number? I'll open an investigation with the carrier and, if it can't be located, arrange a replacement or refund."
    },
    {
      "prompt": "Can I add an item to an order I just placed?",
      "response": "Happy to help if it hasn't shipped yet! Could you share your order number and the item you'd like to add? I'll check whether I can combine it into your existing order."
    },
    {
      "prompt": "I'm allergic to a material, can you confirm what's used?",
      "response": "Thank you for letting me know, your safety matters. Could you tell me the product name? I'll confirm the exact materials so you can decide with confidence."
    },
    {
      "prompt": "The website crashed during my payment.",
      "response": "I'm sorry the site crashed mid-payment, that's worrying. Could you confirm your email and whether you saw a charge? I'll verify whether the order went through and make sure you're not charged incorrectly."
    },
    {
      "prompt": "I want to leave a complaint.",
      "response": "I'm sorry something went wrong, and I want to make sure your complaint is heard. Could you tell me what happened and share your order number? I'll log it formally and do my best to resolve it for you."
    },
    {
      "prompt": "Is gift wrapping available?",
      "response": "Yes, we offer gift wrapping! You can select it at checkout for a small fee. If you've already ordered, share your order number and I'll check whether I can still add it for you."
    },
    {
      "prompt": "My order was canceled without my permission.",
      "response": "I'm so sorry your order was canceled unexpectedly, that's understandably upsetting. Could you share your order number? I'll find out why it happened and help you reorder or restore it as quickly as possible."
    },
    {
      "prompt": "How do I update my payment method?",
      "response": "Happy to guide you! You can update your payment method under 'Account Settings > Payment'. If you run into any issues, let me know and I'll walk you through it step by step."
    },
    {
      "prompt": "The product smells strange out of the box.",
      "response": "I'm sorry to hear that, an unusual smell isn't something you should experience. Could you share your order number? I'll arrange a replacement and flag this with our quality team for review."
    },
    {
      "prompt": "I haven't received an order confirmation email.",
      "response": "I'm sorry the confirmation didn't arrive, let me check for you. Could you share the email you used at checkout? I'll verify your order went through and resend the confirmation right away."
    },
    {
      "prompt": "Can I split my payment across two cards?",
      "response": "Thanks for asking! Unfortunately split payments aren't supported at checkout, but you could use a gift card balance alongside one card. Let me know if you'd like help setting that up."
    },
    {
      "prompt": "Your packaging uses too much plastic.",
      "response": "Thank you for caring about this, I completely understand. I'll pass your feedback to our packaging team, as we're actively working to reduce plastic. I appreciate you taking the time to share it."
    },
    {
      "prompt": "I received a damaged gift and I'm embarrassed.",
      "response": "I'm so sorry, especially since it was a gift, that's really disappointing. Could you share the order number? I'll prioritize a fast replacement so you can give it with confidence, and I'll make this as smooth as possible."
    },
    {
      "prompt": "The checkout page won't load.",
      "response": "I'm sorry the checkout page isn't loading, that's frustrating when you're ready to buy. Could you try a different browser or clear your cache? If it still fails, tell me the items and I'll help place the order for you directly."
    }
  ]
}

demos = DEMOS["data"]
print(f"Loaded {len(demos)} demonstrations.")
print("\nExample:")
print(" Customer:", demos[0]["prompt"])
print(" Agent   :", demos[0]["response"])

## 2. Formatting & train/val split

**What:** Wrap every example in a **fixed chat template** and split 50 / 10 (train / val).

**Why:** The model must see the *same* format during training, generation, and evaluation — otherwise it can't tell where the prompt ends and the answer begins. `build_prompt_only` returns just the prompt (used at inference so the model completes the answer); `build_training_text` adds the response + EOS (used for training).

**How it works:** The `### Customer:` / `### Agent:` markers give the model a consistent structure to learn.

In [ ]:
PROMPT_TEMPLATE = "### Customer:\n{prompt}\n\n### Agent:\n"
EOS = "<|endoftext|>"   # GPT-2 / DistilGPT-2 end-of-text token

def build_prompt_only(prompt: str) -> str:
    """Prompt portion only — used at inference time."""
    return PROMPT_TEMPLATE.format(prompt=prompt)

def build_training_text(prompt: str, response: str) -> str:
    """Full training sequence: prompt + target response + EOS."""
    return build_prompt_only(prompt) + response + EOS

def to_records(items):
    out = []
    for d in items:
        out.append({
            "prompt": d["prompt"],
            "response": d["response"],
            "prompt_only": build_prompt_only(d["prompt"]),
            "text": build_training_text(d["prompt"], d["response"]),
        })
    return out

train_records = to_records(demos[:-10])   # first 50
val_records   = to_records(demos[-10:])   # last 10

print(f"Train: {len(train_records)}  |  Val: {len(val_records)}")
print("\n--- one training text ---")
print(train_records[0]["text"])

## 3. Dataset class with **prompt masking** — the key SFT detail

**What:** A causal-LM dataset where the loss is **masked over the prompt tokens** (set to `-100`).

**Why:** `-100` is PyTorch's "ignore this token in the cross-entropy loss" value. We do **not** want the model to waste capacity memorizing the customer's message — we only want it to learn *how to respond*. So the loss is computed **only** over the agent's reply.

**How it works:** For each example we tokenize the prompt-only part, find its length, and overwrite those positions in `labels` with `-100`. Everything after (the response) keeps its real token id as the target.

**In the paper:** This is the essence of SFT — supervised imitation of the *desired output behavior*. Note: there is **no reward and no RL here** — that's exactly what makes SFT insufficient and motivates Person 2's PPO.

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, records, tokenizer):
        self.records = records
        self.tok = tokenizer

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        prompt_ids = self.tok(rec["prompt_only"], add_special_tokens=False)["input_ids"]
        full_ids   = self.tok(rec["text"], add_special_tokens=False)["input_ids"][:MAX_LEN]

        input_ids = full_ids
        labels = list(full_ids)
        # --- mask the prompt portion so loss is only on the agent reply ---
        n_prompt = min(len(prompt_ids), len(labels))
        for i in range(n_prompt):
            labels[i] = -100
        return {"input_ids": input_ids, "labels": labels}


def collate(batch, pad_id):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids, labels, attn = [], [], []
    for b in batch:
        pad = max_len - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad_id] * pad)
        labels.append(b["labels"] + [-100] * pad)   # padding also ignored
        attn.append([1] * len(b["input_ids"]) + [0] * pad)
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attn),
    }

## 4. Load the pretrained (Base) model & tokenizer

We keep a copy of the **Base** model untouched for the comparison later, and fine-tune a separate copy into the **SFT** model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
pad_id = tokenizer.pad_token_id

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

train_ds = SFTDataset(train_records, tokenizer)
val_ds   = SFTDataset(val_records, tokenizer)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,
                          collate_fn=lambda b: collate(b, pad_id))
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False,
                          collate_fn=lambda b: collate(b, pad_id))
print("Model and dataloaders ready.")

## 5. Supervised fine-tuning loop

**What:** Standard supervised training — forward pass, cross-entropy loss (on the masked labels), backprop, optimizer step.

**Why these choices:**
- **AdamW + cosine LR schedule with warmup** mirrors the paper's cosine learning-rate decay.
- **Gradient clipping (1.0)** for stability on a tiny dataset.
- We track **train loss per step** and **validation loss per epoch** so we can show the model is actually learning.

**How it works:** Because labels are masked, `out.loss` is cross-entropy *only over the agent reply tokens* — the model learns to imitate good responses.

**In the paper:** This is Step 1. The paper trains SFT for several epochs with cosine decay; we do the small-scale equivalent.

In [ ]:
EPOCHS = 8
LR = 5e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps)

@torch.no_grad()
def evaluate_loss(m, loader):
    m.eval(); total, n = 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        total += m(**batch).loss.item(); n += 1
    return total / max(n, 1)

history = {"step": [], "train_loss": [], "epoch": [], "val_loss": []}
step = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        step += 1
        history["step"].append(step); history["train_loss"].append(loss.item())
    vl = evaluate_loss(model, val_loader)
    history["epoch"].append(epoch); history["val_loss"].append(vl)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={loss.item():.4f} "
          f"| val_loss={vl:.4f} | val_ppl={math.exp(vl):.2f}")

sft_model = model   # this is Person 2's frozen reference policy
print("\nSFT training done.")

### Loss curves
The training loss should fall steadily; validation loss may flatten early (the paper notes SFT overfits validation loss after ~1 epoch, yet more epochs still help downstream — same idea here).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history["step"], history["train_loss"], color="#2563eb")
ax1.set_title("SFT training loss (per step)")
ax1.set_xlabel("Step"); ax1.set_ylabel("Loss"); ax1.grid(alpha=0.3)
ax2.plot(history["epoch"], history["val_loss"], marker="o", color="#dc2626")
ax2.set_title("SFT validation loss (per epoch)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Generation helper

**What:** One function that produces an agent reply for a single customer message.

**Why:** We must run Base and SFT under **identical conditions** — same template, same sampling (`temperature=0.7`, `top_p=0.9`) — so the comparison is fair. We also trim any extra `### Customer:` blocks the model hallucinates.

In [ ]:
@torch.no_grad()
def generate_reply(m, tok, customer_prompt, max_new_tokens=80,
                   temperature=0.7, top_p=0.9):
    prompt_text = build_prompt_only(customer_prompt)
    inputs = tok(prompt_text, return_tensors="pt").to(DEVICE)
    out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                     temperature=temperature, top_p=top_p,
                     pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]   # only new tokens
    text = tok.decode(gen, skip_special_tokens=True)
    for marker in ["### Customer:", "### Agent:"]:
        if marker in text:
            text = text.split(marker)[0]
    return text.strip()

## 7. Constraint-following metric

**What:** A small, transparent **rubric** that scores any reply on five constraints (pass/fail = 1/0).

**Why:** The paper claims InstructGPT "reliably follows explicit constraints in the instruction." To measure that concretely I turn it into an explainable, keyword-based rubric (not a black box) so anyone can see *why* a reply passed or failed.

**How it works:** `score_constraints` returns the per-constraint 0/1 dict; `constraint_rate` is the fraction satisfied. This is what feeds the evaluation table.

**In the paper:** Constraint-following is one of the clearest axes where InstructGPT beats the base GPT model.

In [ ]:
_POLITE  = ["please","thank","happy to","glad to","of course","i understand","i'd be","i would be"]
_EMPATHY = ["sorry","apologi","frustrat","understand how","that's not","disappoint","i understand"]
_ASK     = ["order number","could you","can you","please share","please provide","confirm","let me know","tell me"]
_NEXT    = ["refund","replacement","replace","track","arrange","process","ship","exchange","resolve","help","label","update","send"]
_RUDE    = ["i don't know","not my problem","whatever","deal with it","no idea","can't help"]

def _has_any(t, words): return any(w in t for w in words)

def score_constraints(reply):
    t = reply.lower().strip()
    return {
        "polite":           int(_has_any(t, _POLITE)),
        "empathy":          int(_has_any(t, _EMPATHY)),
        "asks_for_info":    int(_has_any(t, _ASK)),
        "offers_next_step": int(_has_any(t, _NEXT)),
        "not_rude":         int(not _has_any(t, _RUDE)),
        "substantive":      int(len(t.split()) >= 5),
    }

def constraint_rate(reply):
    s = score_constraints(reply)
    return sum(s.values()) / len(s)

# sanity check
print(score_constraints("I'm so sorry. Could you share your order number? I'll arrange a refund."))
print(score_constraints("I don't know."))

## 8. Base vs SFT on 20 held-out prompts — the core experiment

**What:** For 20 prompts **not seen in training**, generate a Base reply and an SFT reply, score both.

**Why held-out:** Unseen prompts test real generalization, not memorization. This is the mini version of the paper's Figure 1 result (SFT beats base GPT).

In [ ]:
HELD_OUT_PROMPTS = [
    "My order is stuck in customs.",
    "The screen on my new tablet is cracked.",
    "I was double-billed for shipping.",
    "Can you help me find my tracking number?",
    "The wrong size shoes were delivered.",
    "I've been waiting on hold for an hour.",
    "My loyalty points disappeared from my account.",
    "The fabric started tearing after one wear.",
    "I need to update the email on my account.",
    "My replacement order also arrived damaged.",
    "Do you price match competitors?",
    "The gift card I bought isn't working.",
    "My package was marked delivered but it's missing.",
    "Can I cancel an order placed five minutes ago?",
    "The instructions are in a language I can't read.",
    "I was charged a fee I don't recognize.",
    "How do I return a faulty headphone set?",
    "The promised next-day delivery never came.",
    "My account got locked for no reason.",
    "Can I get an invoice for my business records?",
]

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()

rows, base_rates, sft_rates = [], [], []
for i, prompt in enumerate(HELD_OUT_PROMPTS, 1):
    b = generate_reply(base_model, tokenizer, prompt)
    s = generate_reply(sft_model,  tokenizer, prompt)
    br, sr = constraint_rate(b), constraint_rate(s)
    base_rates.append(br); sft_rates.append(sr)
    rows.append({"id": i, "prompt": prompt, "base_reply": b, "sft_reply": s,
                 "base_breakdown": score_constraints(b),
                 "sft_breakdown": score_constraints(s)})
    print(f"[{i:2d}/20] base={br:.2f}  sft={sr:.2f} | {prompt}")

### Side-by-side examples
This is the slide-killer: show one or two of these out loud. Base tends to drift or stay unhelpful; SFT apologizes, asks for the order number, offers a next step.

In [ ]:
for r in rows[:3]:
    print("="*70)
    print("PROMPT:", r["prompt"])
    print("\n[BASE]", r["base_reply"])
    print("\n[SFT ]", r["sft_reply"])
    print()

### Constraint-following summary table & chart
**In the paper:** quantitative evidence that fine-tuning on human demonstrations improves constraint-following over the base model.

In [ ]:
keys = list(rows[0]["base_breakdown"].keys())
summary = {"constraint": [], "base": [], "sft": []}
for k in keys:
    summary["constraint"].append(k)
    summary["base"].append(mean(r["base_breakdown"][k] for r in rows))
    summary["sft"].append(mean(r["sft_breakdown"][k] for r in rows))
summary["constraint"].append("OVERALL_RATE")
summary["base"].append(mean(base_rates))
summary["sft"].append(mean(sft_rates))

print(f"{'constraint':<18}{'base':>8}{'sft':>8}")
for c, b, s in zip(summary["constraint"], summary["base"], summary["sft"]):
    print(f"{c:<18}{b:>8.3f}{s:>8.3f}")

x = range(len(summary["constraint"])); w = 0.38
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar([i-w/2 for i in x], summary["base"], w, label="Base", color="#9ca3af")
ax.bar([i+w/2 for i in x], summary["sft"],  w, label="SFT",  color="#2563eb")
ax.set_xticks(list(x)); ax.set_xticklabels(summary["constraint"], rotation=30, ha="right")
ax.set_ylabel("Pass rate"); ax.set_title("Constraint-following: Base vs SFT (20 held-out prompts)")
ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

### Blind human-evaluation sheet ("which reply is more natural?")
**Why blind:** I randomize which side (A/B) is SFT per row so the human rater can't tell which model produced which reply — this removes bias. The hidden mapping is stored separately for scoring afterward. This is the small-scale version of the paper's human win-rate evaluation.

In [ ]:
rng = random.Random(SEED)
human_rows = []
for r in rows:
    if rng.random() < 0.5:
        A, B, mapping = r["base_reply"], r["sft_reply"], "A=base,B=sft"
    else:
        A, B, mapping = r["sft_reply"], r["base_reply"], "A=sft,B=base"
    human_rows.append({"id": r["id"], "prompt": r["prompt"],
                       "reply_A": A, "reply_B": B,
                       "winner(A/B/tie)": "", "_mapping": mapping})

print("Example rows of the blind sheet (fill in 'winner' by hand):")
for hr in human_rows[:2]:
    print("-"*60)
    print("PROMPT:", hr["prompt"])
    print(" A:", hr["reply_A"])
    print(" B:", hr["reply_B"])

## 9. Conclusion — and the hand-off to Person 2

**What this section showed:**
- The **base** model produces fluent text but does not reliably follow the *intent* of a support request — the paper's "language-modeling objective is misaligned" point.
- **SFT** sharply improves constraint-following (empathy, asking for the order number, offering a next step) — visible in the table and chart.

**Why SFT is still not enough (the motivation for RLHF):**
SFT only **imitates** demonstrations. It never *optimizes* for what humans actually prefer — it has no notion of "this reply is better than that one." It also can only be as good as the demonstrations it was shown.

> **Hand-off:** That gap is exactly what Person 2 fills — a **Reward Model** learns human preferences from pairwise comparisons, and **PPO** optimizes the SFT policy against that reward (with a **KL penalty against this frozen SFT model** so it stays fluent). Together: Pretraining -> SFT -> Reward Model -> PPO -> aligned LLM.
